# Chunking

**Purpose:** Split cleaned pages into smaller overlapping chunks suitable for embedding and retrieval.

**Pipeline Position:** `data/cleaned/*.json` → **[CHUNKING]** → `data/chunks/chunks.json`

---

### What this notebook does:
1. Loads all cleaned page JSON files from `data/cleaned/`
2. Splits each page's text into overlapping token-based chunks
3. Assigns a unique `chunk_id` and preserves full metadata per chunk
4. Saves all chunks into a single flat `chunks.json` in `data/chunks/`

### Chunking Strategy:
- **Chunk size:** 400 tokens (~300–500 words)
- **Overlap:** 50 tokens (preserves context across chunk boundaries)
- **Tokenizer:** `tiktoken` (cl100k_base — same as OpenAI/most embedding models)

### Output saved to:
`data/chunks/chunks.json` — flat list of all chunks across all documents

---

In [1]:
import json
import tiktoken
from pathlib import Path

print("Libraries imported")

Libraries imported


In [ ]:
CLEANED_DIR  = Path("../../data/cleaned")
CHUNKS_DIR   = Path("../../data/chunks")

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# Chunking Parameters
CHUNK_SIZE    = 400   # tokens per chunk
CHUNK_OVERLAP = 50    # overlap tokens between consecutive chunks
ENCODING_NAME = "cl100k_base"  # tiktoken encoding (works with OpenAI + most embedding models)

# Display cleaned files
cleaned_files = sorted(CLEANED_DIR.glob("*.json"))
print(f"Found {len(cleaned_files)} cleaned files")
for f in cleaned_files:
    print(f"  - {f.name}")

📂 Found 16 cleaned files
  - ABL-Business-User-Guidelines.json
  - ABL-FAQs.json
  - Bank-Alfalah-FAQs-Account-Opening.json
  - Bank-Alfalah-FAQs-Personal-Loan.json
  - Bank-Alfalah-Self-Service-Banking.json
  - HBL-FAQs-Home-Remittance.json
  - HBL-FAQs.json
  - HBL-Islamic-Current-Account.json
  - HBL-Work-Conventional-Accounts.json
  - Meezan-Bank-FAQs-Debit-Cards.json
  - Meezan-Bank-FAQs-Digital-Account.json
  - Meezan-Bank-FAQs-Roshan-Apna-Ghar.json
  - Meezan-Bank-FAQs-Roshan-Digital-Account.json
  - Meezan-Bank-FAQs-Salaried.json
  - State-Bank-FAQs-History.json
  - State-Bank-FAQs.json


In [4]:
# Load the tokenizer
enc = tiktoken.get_encoding(ENCODING_NAME)

"""
Split text into overlapping token-based chunks.

Args:
    text:        Input text to chunk.
    chunk_size:  Max tokens per chunk.
    overlap:     Tokens shared between consecutive chunks.

Returns:
    List of text strings (chunks).
"""

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    tokens = enc.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text_str = enc.decode(chunk_tokens)
        chunks.append(chunk_text_str.strip())
        if end == len(tokens):
            break
        start += chunk_size - overlap  # slide window forward

    return chunks


print("Chunking function ready")
print(f"   Chunk size : {CHUNK_SIZE} tokens")
print(f"   Overlap    : {CHUNK_OVERLAP} tokens")

Chunking function ready
   Chunk size : 400 tokens
   Overlap    : 50 tokens


In [5]:
all_chunks = []
chunk_id_counter = 0

# Chunk each page of each cleaned file and store metadata
for json_file in cleaned_files:
    with open(json_file, "r", encoding="utf-8") as f:
        pages = json.load(f)

    file_chunk_count = 0

    for page in pages:
        text = page["cleaned_text"]
        chunks = chunk_text(text)

        for chunk_index, chunk in enumerate(chunks):
            all_chunks.append({
                "chunk_id":    f"chunk_{chunk_id_counter:05d}",
                "bank_name":   page["bank_name"],
                "source_file": page["source_file"],
                "page_number": page["page_number"],
                "chunk_index": chunk_index,
                "token_count": len(enc.encode(chunk)),
                "text":        chunk
            })
            chunk_id_counter += 1
            file_chunk_count += 1

    print(f"{json_file.stem:<45}  {len(pages):>3} pages  →  {file_chunk_count:>4} chunks")

print(f"\nTotal chunks created: {len(all_chunks)}")

ABL-Business-User-Guidelines                     6 pages  →     9 chunks
ABL-FAQs                                        19 pages  →    35 chunks
Bank-Alfalah-FAQs-Account-Opening                8 pages  →    11 chunks
Bank-Alfalah-FAQs-Personal-Loan                  3 pages  →     3 chunks
Bank-Alfalah-Self-Service-Banking                5 pages  →     5 chunks
HBL-FAQs-Home-Remittance                         2 pages  →     4 chunks
HBL-FAQs                                         5 pages  →     8 chunks
HBL-Islamic-Current-Account                      4 pages  →     9 chunks
HBL-Work-Conventional-Accounts                   4 pages  →     8 chunks
Meezan-Bank-FAQs-Debit-Cards                     2 pages  →     3 chunks
Meezan-Bank-FAQs-Digital-Account                 3 pages  →     5 chunks
Meezan-Bank-FAQs-Roshan-Apna-Ghar                2 pages  →     4 chunks
Meezan-Bank-FAQs-Roshan-Digital-Account          9 pages  →    18 chunks
Meezan-Bank-FAQs-Salaried                        5 

In [6]:
# Show 3 consecutive sample chunks to verify overlap works correctly
print(f"Showing chunks 0, 1, 2 from file: {all_chunks[0]['source_file']}\n")

for i in range(min(3, len(all_chunks))):
    c = all_chunks[i]
    print(f"{'─'*55}")
    print(f"Chunk ID    : {c['chunk_id']}")
    print(f"Bank        : {c['bank_name']}")
    print(f"Source File : {c['source_file']}")
    print(f"Page        : {c['page_number']}")
    print(f"Token Count : {c['token_count']}")
    print(f"Text Preview: {c['text'][:300]}...")
    print()

Showing chunks 0, 1, 2 from file: ABL-Business-User-Guidelines.pdf

───────────────────────────────────────────────────────
Chunk ID    : chunk_00000
Bank        : Allied Bank Limited (ABL)
Source File : ABL-Business-User-Guidelines.pdf
Page        : 2
Token Count : 187
Text Preview: OBDX – General Guidelines 
Table of Contents 
1. 
1.1. 
1.2. 
1.3. 
2. 
3. 
3.1. 
3.2. 
3.3. 
4. 
Page- 2 
 
 
 
myABL Verify (Soft Token App) Configuration ....................................................................... 3 
Introduction ..........................................................

───────────────────────────────────────────────────────
Chunk ID    : chunk_00001
Bank        : Allied Bank Limited (ABL)
Source File : ABL-Business-User-Guidelines.pdf
Page        : 3
Token Count : 400
Text Preview: OBDX – General Guidelines 
1. myABL Verfiy (Soft Token App) Configuration 
1.1. Introduction 
myABL Verify (Soft Token App) generates one-time password (OTP) on your device for two-factor 
auth

In [ ]:
# Save all chunks to a single JSON file
output_path = CHUNKS_DIR / "chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(all_chunks)} chunks to: {output_path}")

💾 Saved 207 chunks to: ..\data\chunks\chunks.json


In [8]:
# Basic statistics about the chunks
token_counts = [c["token_count"] for c in all_chunks]

print("=" * 45)
print(f"Total chunks         : {len(all_chunks)}")
print(f"Avg tokens per chunk : {sum(token_counts)/len(token_counts):.1f}")
print(f"Min tokens           : {min(token_counts)}")
print(f"Max tokens           : {max(token_counts)}")
print("=" * 45)

# Per-bank breakdown
from collections import Counter
bank_counts = Counter(c["bank_name"] for c in all_chunks)
print("\nChunks per bank:")
for bank, count in bank_counts.most_common():
    print(f"  {bank:<35} : {count}")

Total chunks         : 207
Avg tokens per chunk : 292.4
Min tokens           : 35
Max tokens           : 402

Chunks per bank:
  State Bank of Pakistan (SBP)        : 76
  Allied Bank Limited (ABL)           : 44
  Meezan Bank                         : 39
  Habib Bank Limited (HBL)            : 29
  Bank Alfalah                        : 19
